# Feature panel and labels

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd() / 'scratchpad/hyperliquid-ic' if (Path.cwd() / 'scratchpad/hyperliquid-ic').exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
ARTIFACTS = PROJECT_DIR / '_artifacts-rewrite'
ARTIFACTS.mkdir(exist_ok=True)
from ic_research import ResearchConfig, feature_names, generate_research_panels

CONFIG = ResearchConfig()
observations = pd.read_parquet(ARTIFACTS / 'observations.parquet')
metadata = pd.read_csv(ARTIFACTS / 'vault-metadata.csv', index_col=0)
metadata.index = metadata.index.astype(str).str.lower()
btc_reference = pd.read_parquet(ARTIFACTS / 'btc-reference.parquet')
features, labels, eligibility = generate_research_panels(observations, metadata, btc=btc_reference, config=CONFIG)
features.to_parquet(ARTIFACTS / 'features.parquet', index=False)
labels.to_parquet(ARTIFACTS / 'labels.parquet', index=False)
eligibility.to_parquet(ARTIFACTS / 'eligibility.parquet', index=False)
print('Features:', features.shape, '| labels:', labels.shape, '| daily decisions:', features.date.nunique())


Feature panel: 50/602 vaults; 7,031 eligible daily rows


Feature panel: 100/602 vaults; 16,004 eligible daily rows


Feature panel: 150/602 vaults; 24,670 eligible daily rows


Feature panel: 200/602 vaults; 33,291 eligible daily rows


Feature panel: 250/602 vaults; 43,577 eligible daily rows


...

Feature panel: 550/602 vaults; 96,593 eligible daily rows


Feature panel: 600/602 vaults; 104,267 eligible daily rows


Feature panel: 602/602 vaults; 104,618 eligible daily rows


/Users/moo/code/getting-started/scratchpad/hyperliquid-ic/ic_research.py:375: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[f"{column}__missing"] = features[column].isna().astype(float)
/Users/moo/code/getting-started/scratchpad/hyperliquid-ic/ic_research.py:375: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[f"{column}__missing"] = features[column].isna().astype(float)
/Users/moo/code/getting-started/scratchpad/hyperliquid-ic/ic_research.py:375: PerformanceWarning: DataFrame is highly fragmented.  This is usually t

Features: (104618, 410) | labels: (104618, 94) | daily decisions: 545


In [2]:
reserved = {'date', 'decision_ts', 'address', 'eligible', 'nav', 'tvl_current', 'last_observation_ts'}
manifest = pd.DataFrame({'feature': feature_names(features)})
manifest['missingness_flag'] = manifest.feature.str.endswith('__missing')
manifest['lookback_or_span'] = manifest.feature.str.extract(r'_(\d+)(?:__missing)?$')[0].astype('Int64')
manifest['causal_rule'] = 'fresh observations at or before the completed daily close; no written_at filter'
manifest['missingness_rule'] = 'feature-local; missing values are retained with training-fold imputation'
manifest.to_csv(ARTIFACTS / 'feature-manifest.csv', index=False)
funnel = eligibility.groupby('date').agg(vault_rows=('address', 'size'), two_marks=('has_two_marks', 'sum'), eligible=('eligible', 'sum')).reset_index()
funnel.to_parquet(ARTIFACTS / 'eligibility-funnel.parquet', index=False)
label_summary = []
for horizon in CONFIG.forecast_horizons:
    outcome = f'outcome_id_{horizon}'
    label_summary.append({'horizon': horizon, 'daily_prediction_rows': len(features), 'growth_labels': int(labels[f'forward_log_growth_{horizon}'].notna().sum()), 'daily_variance_labels': int(labels[f'forward_variance_{horizon}'].notna().sum()), 'unique_outcomes': int(labels[outcome].nunique())})
pd.DataFrame(label_summary).to_csv(ARTIFACTS / 'label-coverage.csv', index=False)
display(pd.DataFrame(label_summary))
display(features.groupby('date').size().describe().to_frame('eligible_vaults'))


,horizon,daily_prediction_rows,growth_labels,daily_variance_labels,unique_outcomes
0,7,104618,101066,59694,66127
1,14,104618,99231,58501,64298
2,21,104618,97387,57300,62436
3,30,104618,93360,55467,59716
4,45,104618,89060,51619,55303
5,60,104618,84582,47831,50756
6,90,104618,75695,40014,41993


,eligible_vaults
count,545.000000
mean,191.959633
std,76.113561
min,69.000000
25%,124.000000
50%,204.000000
75%,266.000000
max,291.000000


In [3]:
# Acceptance check: daily decisions work for a weekly vault after two marks,
# but no daily-risk feature or target is created from the unobserved path.
synthetic = pd.DataFrame({
    'address': ['0xweekly'] * 5,
    'timestamp': pd.to_datetime(['2026-01-01', '2026-01-08', '2026-01-15', '2026-01-22', '2026-02-15']),
    'written_at': pd.to_datetime(['2026-02-16'] * 5),
    'share_price': [1.0, 1.02, 1.01, 1.03, 1.04],
    'total_assets': [10000.] * 5,
    'is_fresh': [True] * 5,
})
synthetic_btc = pd.DataFrame({'close': 100.0}, index=pd.date_range('2025-12-01', '2026-02-15', freq='D'))
synthetic_features, synthetic_labels, _ = generate_research_panels(synthetic, pd.DataFrame(index=['0xweekly']), btc=synthetic_btc, config=CONFIG)
between_updates = synthetic_features[synthetic_features.date.between('2026-01-08', '2026-01-14')]
assert len(between_updates) == 7
assert between_updates.daily_vol_7.isna().all()
assert synthetic_labels.forward_variance_7.isna().all()
assert (between_updates.nav_age_days <= 14).all()
print('Weekly sparse-observation acceptance check passed:', len(between_updates), 'daily predictions between marks')


Feature panel: 1/1 vaults; 28 eligible daily rows
Weekly sparse-observation acceptance check passed: 7 daily predictions between marks


/Users/moo/code/getting-started/scratchpad/hyperliquid-ic/ic_research.py:375: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[f"{column}__missing"] = features[column].isna().astype(float)
/Users/moo/code/getting-started/scratchpad/hyperliquid-ic/ic_research.py:375: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[f"{column}__missing"] = features[column].isna().astype(float)
/Users/moo/code/getting-started/scratchpad/hyperliquid-ic/ic_research.py:375: PerformanceWarning: DataFrame is highly fragmented.  This is usually t